In [ ]:
#File Merger 1996-1999
import os
import pandas as pd

# Define directories
input_directory = r"_raw"
output_directory = r"_raw_merged"
output_filename = "CRE-4-1999_raw.csv"

# List to store main format data
main_data_frames = []
# List to store annex format data
annex_data_frames = []

# Function to extract date from filename
def extract_date(filename):
    parts = filename.split('-')
    if len(parts) > 4:
        return f"{parts[2]}-{parts[3]}-{parts[4]}"
    return None

# Loop through each file in the directory
file_count = 0
for filename in os.listdir(input_directory):
    if filename.endswith(".csv"):
        file_path = os.path.join(input_directory, filename)
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Extract date from filename
        date = extract_date(filename)
        
        # Check if it's the main format or annex format
        if "ANN" in filename:
            # Add the annex data to a separate list and tag the columns with a prefix
            df = df.add_prefix('Annex_')
            df['Date'] = date  # Add date for proper merging later
            annex_data_frames.append(df)
        else:
            # Add date column to the main format files and append to list
            df['Date'] = date
            main_data_frames.append(df)
        
        # Increment the file count and print progress
        file_count += 1
        print(f"Merged file {file_count}: {filename}")

# Combine all main format data into a single DataFrame by stacking (appending vertically)
merged_main_df = pd.concat(main_data_frames, ignore_index=True)

# Combine all annex format data into a separate DataFrame if annex data exists
if annex_data_frames:
    merged_annex_df = pd.concat(annex_data_frames, ignore_index=True)
    # Merge the main and annex dataframes on their common 'Date' column
    merged_df = pd.merge(merged_main_df, merged_annex_df, on='Date', how='left')
else:
    merged_df = merged_main_df

# Save the merged DataFrame to CSV
output_path = os.path.join(output_directory, output_filename)
merged_df.to_csv(output_path, index=False)

# Final print statement with the total count
print(f"\nAll files merged. Total files merged: {file_count}")
print(f"Merged CSV saved at {output_path}")



In [ ]:
#File Merger 1999-2024
import os
import pandas as pd
import re

# Define the input and output directories
source_dir = r"_raw"
output_dir = r"_raw_merged"
output_file = os.path.join(output_dir, "CRE-9-2024_raw.csv")

# Check if the output directory exists, if not, create it
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Output directory created at {output_dir}")

# List to store individual dataframes
dataframes = []

# Regex pattern to extract the date from the filename
date_pattern = re.compile(r'(\d{4}-\d{2}-\d{2})')

# Variable to keep track of the number of files merged
file_count = 0

# Process each CSV file in the source directory
for file_name in os.listdir(source_dir):
    if file_name.endswith(".csv"):
        file_path = os.path.join(source_dir, file_name)
        
        try:
            # Extract date from the filename
            date_match = date_pattern.search(file_name)
            if date_match:
                date_str = date_match.group(1)
            else:
                print(f"Date not found in filename: {file_name}")
                continue  # Skip files without a valid date

            # Read the CSV file
            df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')

            # Add the date column to the dataframe
            df['date'] = date_str

            # Append the dataframe to the list
            dataframes.append(df)

            # Increment file count
            file_count += 1

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

# Concatenate all dataframes into one
if dataframes:
    merged_df = pd.concat(dataframes, ignore_index=True)

    # Save the merged dataframe to the output file
    merged_df.to_csv(output_file, index=False, encoding='utf-8')

    # Print the number of files merged
    print(f"\nData merged from {file_count} files and saved to {output_file}")

else:
    print("No valid CSV files found to merge.")

